# 01. Exploring embeddings

**Goal:** show how text becomes numbers, and how close numbers mean close meaning.
This is the idea behind the job search in SmartHire.

In [1]:
import os, sys
os.environ["TQDM_DISABLE"] = "1"     # hide progress bars so the notebook stays tidy
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("Working folder:", ROOT)

Working folder: C:\Users\pramod\Smarthire-genai


## Step 1: turn sentences into numbers
Each sentence becomes a list of numbers. Sentences with a similar meaning get similar numbers.

In [2]:
import pandas as pd
from src.search.embed import embed_texts

sentences = [
    "Data Analyst with SQL and Power BI",
    "Business intelligence developer who builds dashboards in DAX",
    "React frontend developer building web pages",
    "Chef who cooks Italian food",
    "Machine learning engineer training neural networks",
]
vectors = embed_texts(sentences)
print("Each sentence became a list of", vectors.shape[1], "numbers")
print("First 8 numbers of sentence 1:", vectors[0][:8].round(3))

C:\Users\pramod\Smarthire-genai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 53.35it/s]

Each sentence became a list of 384 numbers
First 8 numbers of sentence 1: [ 0.011  0.004 -0.084  0.072 -0.109 -0.023  0.048  0.013]


## Step 2: measure how close each pair is
The numbers are scaled so that multiplying them gives a similarity score (1 means the same meaning, 0 means unrelated).

In [3]:
sim = vectors @ vectors.T
table = pd.DataFrame(sim.round(2),
                     index=[f"S{i+1}: {s[:30]}" for i, s in enumerate(sentences)],
                     columns=[f"S{i+1}" for i in range(len(sentences))])
table

,S1,S2,S3,S4,S5
S1: Data Analyst with SQL and Powe,1.00,0.47,0.01,-0.05,0.15
S2: Business intelligence develope,0.47,1.00,0.11,0.06,0.19
S3: React frontend developer build,0.01,0.11,1.00,-0.00,0.09
S4: Chef who cooks Italian food,-0.05,0.06,-0.00,1.00,0.13
S5: Machine learning engineer trai,0.15,0.19,0.09,0.13,1.00


## Step 3: meaning is not the same as shared words
Sentences 1 and 2 share no words, yet they are close in meaning. A chef is far from both.

In [4]:
a, b = sentences[0], sentences[1]
shared = set(a.lower().split()) & set(b.lower().split())
print("Shared words between S1 and S2:", shared or "none")
print("Meaning similarity S1 and S2:", round(float(sim[0][1]), 2))
print("Meaning similarity S1 and S4 (chef):", round(float(sim[0][3]), 2))

Shared words between S1 and S2: none
Meaning similarity S1 and S2: 0.47
Meaning similarity S1 and S4 (chef): -0.05


## Step 4: the same idea on real jobs
SmartHire turns every job into numbers once, then finds the jobs closest to a student's profile.

In [5]:
from src.search.job_search import search_jobs

for r in search_jobs("Python SQL Excel Power BI data analyst fresher", top_n=5):
    print(r["score"], "|", r["Job_Title"], "|", r["Skills_Required"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 141.73it/s]

0.583 | Data Analyst | Power BI, SQL, Data Visualization, Python, Tableau, Excel
0.578 | Power BI Developer | DAX, Data Modeling, Power BI, SQL, ETL, Excel
0.577 | Power BI Developer | Data Modeling, SQL, DAX, Excel
0.576 | Power BI Developer | SQL, ETL, Data Modeling, Power BI, Excel, Azure
0.574 | Power BI Developer | Power BI, Excel, SQL, ETL, Data Modeling


## What I learned
- Meaning search finds related jobs even when the words differ.
- It can miss exact tool names, which is why the app also uses keyword search.